# Project Part 1: Data Pipeline — Cleaning & Sentiment Analysis

**Run this notebook once before starting Part 2 (Whitepaper).**

This notebook takes your school's raw geoparsed data, applies your team's location review, runs sentiment analysis, and exports files that every subsequent notebook depends on. Each processed dataset is saved in two formats: `.csv` for inspection and `.pickle` to preserve dtypes for Plotly maps.

| Output files | Used by |
|---|---|
| `data/{SCHOOL}/{SCHOOL}_geoparsed_long_cleaned.csv` / `.pickle` | Part 2 (Whitepaper) before/after map |
| `data/{SCHOOL}/{SCHOOL}_geoparsed_long_cleaned_sentiment.csv` / `.pickle` | Part 2 (Whitepaper) sentiment maps, Part 3 Base Map, Part 4 (Flythrough) |

The notebook ends with a git branch → commit → pull request workflow to submit your cleaned data for grading.

## Overview

- **Prerequisites:** Your school's geoparsed data exists in `data/{SCHOOL}/` (generated by the instructor pipeline)
- **Parts:**
  1. Load raw geoparsed data
  2. Review locations in Google Sheets
  3. Load and validate the review sheet from Google Sheets
  4. Apply corrections and export the cleaned data file
  5. Run RoBERTa sentiment analysis and export results
  6. Commit and push to GitHub

📋 **Instructions:**

Fill in `SCHOOL` below and run the cell. You will fill in `SHEETS_URL` later, after completing the Google Sheets review in section 2.

1. **`SCHOOL`** — your assigned school abbreviation. Must match the folder name exactly.
   - Valid values: `GMU`, `ODU`, `UNC`, `UVA`, `VCU`, `VirginiaTech`, `WM`
2. **`SHEETS_URL`** — leave this as the placeholder for now. After completing section 2, paste your published CSV URL here and re-run this cell.

> 👉 **Note:** *Every cell in this notebook uses `SCHOOL` automatically — you only need to set it once here.*

In [1]:
SCHOOL     = "UNC"                      # ← replace: GMU, ODU, UNC, UVA, VCU, VirginiaTech, WM
SHEETS_URL = "PASTE_YOUR_PUBLISHED_CSV_URL_HERE"  # ← fill in after completing section 2

# ── Derived paths (do not change) ─────────────────────────────────────────
RAW_PATH       = f'../data/{SCHOOL}/{SCHOOL}_geoparsed_long.csv'
CLEANED_PATH   = f'../data/{SCHOOL}/{SCHOOL}_geoparsed_long_cleaned'
SENTIMENT_PATH = f'../data/{SCHOOL}/{SCHOOL}_geoparsed_long_cleaned_sentiment'

print(f'School      : {SCHOOL}')
print(f'Raw data    : {RAW_PATH}')
print(f'Cleaned     : {CLEANED_PATH}.csv / .pickle')
print(f'Sentiment   : {SENTIMENT_PATH}.csv / .pickle')

School      : UNC
Raw data    : ../data/UNC/UNC_geoparsed_long.csv
Cleaned     : ../data/UNC/UNC_geoparsed_long_cleaned.csv / .pickle
Sentiment   : ../data/UNC/UNC_geoparsed_long_cleaned_sentiment.csv / .pickle


## 📖 1 Follow Along — Load Raw Geoparsed Data

You do not need to write or modify any code in this section. Run each cell and focus on understanding what the code is doing and why.

The raw geoparsed file was created by the instructor pipeline. It contains every sentence from your school's subreddit that mentioned a place name, with the geoparser's best guess at coordinates. Your review sheet contains your team's corrections — this notebook will merge the two.

In [2]:
import sys
import pandas as pd

sys.path.insert(0, "..")
sys.path.insert(0, "../lesson_5_sentiment_analysis")
from tests.helpers import load_and_validate_review_sheet, apply_review_corrections
from sentiment_utils import add_sentiment_to_column

df_raw = pd.read_csv(RAW_PATH)
print(f'Loaded {len(df_raw):,} rows  |  {df_raw["place"].nunique():,} unique places')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head(3)

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 1,263 rows  |  370 unique places
Columns: ['type', 'date', 'score', 'sentences', 'year_month', 'place', 'latitude', 'longitude', 'feature_type', 'admin1_name', 'admin2_name', 'country_name', 'school', 'place_type', 'action', 'corrected_name', 'corrected_latlon', 'corrected_place_type', 'reviewer', 'place_count']


,type,date,score,sentences,year_month,place,latitude,longitude,feature_type,admin1_name,admin2_name,country_name,school,place_type,action,corrected_name,corrected_latlon,corrected_place_type,reviewer,place_count
0,comment,2025-10-01,13,This is a lot of why I decided not to stay in ...,2025-10,North Carolina,35.50069,-80.00032,first-order administrative division,North Carolina,NaN,United States,UNC,State,KEEP,NaN,NaN,NaN,NaN,132
1,comment,2023-09-15,-3,NC has an open carry gun law.,2023-09,North Carolina,35.50069,-80.00032,first-order administrative division,North Carolina,NaN,United States,UNC,State,KEEP,NaN,NaN,NaN,NaN,132
2,comment,2025-02-09,19,Thom Tillis' office is at 310 New Bern Ave Sui...,2025-02,North Carolina,35.50069,-80.00032,first-order administrative division,North Carolina,NaN,United States,UNC,State,KEEP,NaN,NaN,NaN,NaN,132


> 📊 **Output:** You should see the raw geoparsed rows. The `action`, `corrected_name`, `corrected_latlon`, `corrected_place_type`, and `reviewer` columns will be empty — your team fills them in via Google Sheets.

## 2 Review in Google Sheets

For this section, you will repeat the steps from [Lesson 4.4](../lesson_4_finding_locations/lesson_4_4_preparing_review_sheet.ipynb). Please consult this lesson for more thorough documentation.
**One student** starts this step; the whole team contributes. You will import your school's raw data into Google Sheets, review each location as a team, and publish the sheet so this notebook can load the result in section 3.

### 2.1 Import the file

1. Download the file: `data/{SCHOOL}/{SCHOOL}_geoparsed_long.csv` to a local directory
1. Go to [sheets.google.com](https://sheets.google.com) → **Blank spreadsheet**
2. **File → Import → Upload** → select `data/{SCHOOL}/{SCHOOL}_geoparsed_long.csv` from your local machine
3. Choose **Replace spreadsheet**, separator type **Comma**
4. Rename the spreadsheet: `{SCHOOL}_geoparsed_long_cleaned`
6. **Share → Anyone with the link → Editor** → copy the link and post it in your team channel

> 👉 **Note:** *Sort by `place_count` descending first — high-frequency places matter most and are worth checking carefully.*

### 2.2 Add data validation

Set up three dropdown validations so the whole team fills in consistent values.

**Column `action`** *(most important)*:

1. Click the `action` column header to select the whole column
2. **Data → Data validation → Add rule**
3. Criteria: **Dropdown** → add three options: `KEEP`, `CORRECT`, `REMOVE`
4. "If data is invalid": **Reject input**
5. Right-click the `action` header cell → **Insert note** → paste: *KEEP = location is correct. CORRECT = right place, wrong details. REMOVE = not a real location or geoparser error.*

**Column `reviewer`**:

1. Select the `reviewer` column → **Data → Data validation → Add rule**
2. Criteria: **Dropdown** → add each team member's name

**Column `corrected_place_type`**:

1. Select the `corrected_place_type` column → **Data → Data validation → Add rule**
2. Criteria: **Dropdown** → add: `Country`, `State`, `Region`, `City`, `Neighborhood`, `University`, `Road`, `Building`, `Natural Feature`

### 2.3 Publish the sheet and set `SHEETS_URL`

Once your team has reviewed all rows:

1. **File → Share → Publish to web**
2. Choose: **Entire document** → **Comma-separated values (.csv)**
3. Click **Publish** → copy the URL
4. Create a *new branch*
4. Scroll back up to the first code cell, paste the URL into `SHEETS_URL`, and re-run that cell

### 2.4 Review the rows

Work through the rows as a team. For each location:

- Read the `sentences` column to understand the context
- Check `place`, `latitude`, `longitude`, and `place_type`
- Set `action` to `KEEP`, `CORRECT`, or `REMOVE`
- If `CORRECT`: fill in only the columns that need changing:
  - `corrected_name` — new place name
  - `corrected_latlon` — paste directly from Google Maps (e.g. `38.433998, -78.872973`)
  - `corrected_place_type` — select from the dropdown
- Select your name in `reviewer`

> 👉 **Note:** *Getting coordinates from Google Maps: Right-click any spot on the map → click the coordinates at the top of the menu → they copy automatically. Paste the full string (`38.433998, -78.872973`) into `corrected_latlon` — the comma is fine, Python will split it on import.*



## 📖 3 Follow Along — Load and Validate the Review Sheet

You do not need to write or modify any code in this section. Run each cell and focus on understanding what the code is doing and why.

This step loads your team's completed review from Google Sheets, checks for common errors (wrong action values, CORRECT rows missing coordinates), and prints a progress summary.

In [ ]:
df_review = load_and_validate_review_sheet(SHEETS_URL)

if df_review is None:
    print("\n⛔ Fix the issues above before continuing.")
else:
    reviewed = df_review["action"].isin(["KEEP", "CORRECT", "REMOVE"]).sum()
    pct = reviewed / len(df_review) * 100
    print(f'\nProgress: {reviewed:,} / {len(df_review):,} rows reviewed ({pct:.0f}%)')
    if pct < 100:
        print("⚠️  Not all rows have been reviewed. Continue in Google Sheets before exporting.")

> 👉 **Note:** *If you see validation warnings, fix them in Google Sheets and re-run the cell above. Do not proceed until all warnings are resolved — the cleaned file will be incorrect otherwise.*

## 📖 4 Follow Along — Apply Corrections and Export

You do not need to write or modify any code in this section. Run each cell and focus on understanding what the code is doing and why.

This step does three things:
1. Saves the reviewed sheet back to `{SCHOOL}_geoparsed_long.csv` so your review columns are preserved for grading
2. Applies all corrections (drops REMOVE rows, overwrites coordinates and place names for CORRECT rows)
3. Strips the review columns and saves the clean result as both `{SCHOOL}_geoparsed_long_cleaned.csv` (for inspection) and `{SCHOOL}_geoparsed_long_cleaned.pickle` (for downstream notebooks)

In [ ]:
if df_review is None:
    print("⛔ Cannot export — fix validation errors first (re-run Part 2).")
    df_cleaned = None
else:
    # 1. Save reviewed data back to long CSV (preserves review columns for grading)
    keep_cols = [c for c in df_review.columns if c not in ["corrected_lat", "corrected_lon"]]
    df_review[keep_cols].to_csv(RAW_PATH, index=False)
    print(f'✅ Review columns saved back → {RAW_PATH}')

    # 2. Apply corrections
    df_corrected = apply_review_corrections(df_review)

    # 3. Drop review columns and save cleaned CSV + pickle
    _review_cols = ["action", "corrected_name", "corrected_latlon", "corrected_lat",
                    "corrected_lon", "corrected_place_type", "reviewer", "place_count"]
    df_cleaned = df_corrected.drop(columns=[c for c in _review_cols if c in df_corrected.columns])
    df_cleaned.to_csv(f'{CLEANED_PATH}.csv', index=False)
    df_cleaned.to_pickle(f'{CLEANED_PATH}.pickle')

    print(f'✅ Cleaned CSV    saved → {CLEANED_PATH}.csv')
    print(f'✅ Cleaned pickle saved → {CLEANED_PATH}.pickle')
    print(f'   {len(df_review):,} rows in  →  {len(df_cleaned):,} rows out  '
          f'({len(df_review) - len(df_cleaned):,} REMOVE rows dropped)')
    print(f'   Unique places: {df_cleaned["place"].nunique():,}')

> 📊 **Output:** The summary shows how many rows were dropped (REMOVE) and how many unique place names remain. A typical cleanup removes 5–15% of rows.

## 📖 4.1 Follow Along — Compare Raw vs. Cleaned Locations

You do not need to write or modify any code in this section. Run each cell and focus on understanding what the code is doing and why.

Re-run the cell below at any point to compare the raw geoparsed locations against the cleaned result. The maps read directly from the saved files, so they always reflect the latest state of your work — even after a kernel restart.

In [ ]:
import os
import plotly.express as px

def _place_counts(df):
    return (
        df.groupby(["place", "latitude", "longitude", "place_type"], dropna=True)
        .size()
        .reset_index(name="count")
    )

# Always reload from disk so this reflects the latest saved state
_df_raw = pd.read_csv(RAW_PATH)
raw_places = _place_counts(_df_raw)

fig_raw = px.scatter_map(
    raw_places,
    lat="latitude", lon="longitude",
    color="place_type",
    size="count", size_max=20,
    hover_name="place",
    hover_data={"count": True, "place_type": True, "latitude": False, "longitude": False},
    zoom=5, height=500,
    title=f'{SCHOOL} — Raw data  ({len(_df_raw):,} rows · {len(raw_places):,} unique places)',
    map_style="open-street-map",
)
fig_raw.show()

if os.path.exists(f'{CLEANED_PATH}.pickle'):
    _df_clean = pd.read_pickle(f'{CLEANED_PATH}.pickle')
    clean_places = _place_counts(_df_clean)

    fig_clean = px.scatter_map(
        clean_places,
        lat="latitude", lon="longitude",
        color="place_type",
        size="count", size_max=20,
        hover_name="place",
        hover_data={"count": True, "place_type": True, "latitude": False, "longitude": False},
        zoom=5, height=500,
        title=f'{SCHOOL} — Cleaned data  ({len(_df_clean):,} rows · {len(clean_places):,} unique places)',
        map_style="open-street-map",
    )
    fig_clean.show()
else:
    print("⚠️  Cleaned file not found — run Part 4 first to generate it.")

## 📖 5 Follow Along — Run Sentiment Analysis

You do not need to write or modify any code in this section. Run each cell and focus on understanding what the code is doing and why.

This step runs every sentence in your cleaned dataset through the RoBERTa sentiment model and adds four score columns: `roberta_neg`, `roberta_neu`, `roberta_pos`, and `roberta_compound`.

> 👉 **Note:** *This step takes **10–25 minutes** depending on how many sentences your school has. A progress bar will appear. Do not interrupt the kernel — if it fails, re-run from this cell.*

In [ ]:
if df_cleaned is None:
    print("⛔ No cleaned data available — complete Part 3 first.")
    df_sentiment = None
else:
    print(f'Running sentiment analysis on {len(df_cleaned):,} rows...')
    df_sentiment = add_sentiment_to_column(df_cleaned, "sentences")
    print(f'\nDone. New columns: {[c for c in df_sentiment.columns if c not in df_cleaned.columns]}')

In [ ]:
if df_sentiment is None:
    print("⛔ No sentiment data — run the cell above first.")
else:
    df_sentiment.to_pickle(f'{SENTIMENT_PATH}.pickle')
    df_sentiment.to_csv(f'{SENTIMENT_PATH}.csv', index=False)
    print(f'✅ Saved → {SENTIMENT_PATH}.pickle')
    print(f'✅ Saved → {SENTIMENT_PATH}.csv')
    print(f'\nSentiment distribution (compound score):')
    print(f'  Positive (≥ 0.05): {(df_sentiment["roberta_compound"] >= 0.05).sum():,}')
    print(f'  Neutral  (−0.05 – 0.05): '
          f'{((df_sentiment["roberta_compound"] > -0.05) & (df_sentiment["roberta_compound"] < 0.05)).sum():,}')
    print(f'  Negative (≤ −0.05): {(df_sentiment["roberta_compound"] <= -0.05).sum():,}')

> 📊 **Output:** The sentiment distribution shows the overall emotional tone of what people post about locations at your school. Does one category dominate? Is that what you expected?

> 💡 **Reflection:** Compare your school's sentiment distribution to JMU's when you look at the whitepaper maps. Do schools with more off-campus discussions show different patterns than schools focused on campus life?

## 6 Commit and Push

**One student on the team** does this step after all cells above have run without errors.

### What you are committing

| File | Why |
|---|---|
| `data/{SCHOOL}/{SCHOOL}_geoparsed_long.csv` | Updated with your team's review columns (needed for grading) |
| `data/{SCHOOL}/{SCHOOL}_geoparsed_long_cleaned.csv` | Cleaned data for whitepaper maps (human-readable) |
| `data/{SCHOOL}/{SCHOOL}_geoparsed_long_cleaned.pickle` | Cleaned data with dtypes preserved (used by downstream notebooks) |
| `data/{SCHOOL}/{SCHOOL}_geoparsed_long_cleaned_sentiment.csv` | Sentiment-scored data (human-readable) |
| `data/{SCHOOL}/{SCHOOL}_geoparsed_long_cleaned_sentiment.pickle` | Sentiment-scored data with dtypes preserved (used by all maps) |

### Steps

1. Click **main** in the status bar (bottom left) → **Create new branch…** → name it `{school}-data-pipeline` → **Enter**
2. Open **Source Control** (<img src="../lesson_assets/images/vscode/source-control.svg" alt="Source Control icon" style="height: 14pt; vertical-align: middle;">) → stage only the five files listed above using **+**
3. Click the ✨ sparkle button in the message box to AI-generate a commit message — you may need to click it twice the first time
4. Click the dropdown arrow next to **Commit** → select **Commit & Sync**
5. Click **Publish Branch** → select `origin` if prompted
6. Hover over the commit area in **Source Control** → click the **Create Pull Request** icon → base: `main`, compare: `{school}-data-pipeline` → **Create**
7. On GitHub: **Create Merge Commit** → **Delete branch**
8. Switch back to `main` in the status bar → click **Sync Changes** in **Source Control**

> 👉 **Note:** *Do not stage any other files. If unexpected files appear under Changes, discard them before staging.*

> 👉 **Note:** *If you haven't done this workflow before, see [Lesson 1.1](../lesson_1_the_team/lesson_1_1_git_and_pull_requests.ipynb) for detailed screenshots of every step.*

In [ ]:
> 👉 **Note:** *The auto-generated message should describe your school's data files. If it is unclear or too generic, edit it to something like `data: add {SCHOOL} cleaned and sentiment files` before committing.*

### Pull Request description

In the pull request description, note:
- How many rows were removed from the raw data
- Any interesting corrections your team made (e.g., major geoparser errors you caught)

Request a review from a teammate before merging.

> 👉 **Note:** *After the PR is merged, switch back to `main` in the status bar and click **Sync Changes** in **Source Control** to bring your local copy up to date before starting Part 2.*

---

➡️ **Next:** [Project Part 2 — Whitepaper](project_part_2_whitepaper.ipynb)